

Install required libraries and mount Google Drive for persistent storage.


In [ ]:
# Install dependencies for training, RAG vector indexing, and bot deployment
!pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes pyTelegramBotAPI huggingface_hub sentence-transformers pypdf

from google.colab import drive
import os
import torch

# Mount Google Drive to persist checkpoints and access local files
drive.mount('/content/drive')

# Detect available compute device and set precision
print("\nHardware status:")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    print(f"Active GPU: {gpu_name}")
    print(f"Total VRAM: {total_vram:.2f} GB")
    print(f"Selected compute dtype: {compute_dtype}")
else:
    compute_dtype = torch.float32
    print("Warning: Running on CPU. Switch runtime type to GPU (T4 or higher) for training.")



Log in to Hugging Face to download the gated Meta-Llama-3-8B-Instruct weights.


In [ ]:
import os
from huggingface_hub import login

# Read token from environment variable or pass your read-access token
hf_token = os.getenv("HF_TOKEN")

if hf_token:
    login(token=hf_token)
    print("Hugging Face login complete.")
else:
    print("Please set your HF_TOKEN to access Llama-3.")




Central configuration for training paths, RAG knowledge directory, and LoRA parameters.


In [ ]:
import os

# Model identifiers
MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# File system paths on Google Drive
GDRIVE_BASE = "/content/drive/MyDrive/LLM_Training"
DATASET_PATH = os.path.join(GDRIVE_BASE, "master.jsonl")            # Training dataset file
RAG_DATA_DIR = os.path.join(GDRIVE_BASE, "academic_knowledge_base") # Directory containing PDFs, txt, md syllabus files
OUTPUT_DIR = os.path.join(GDRIVE_BASE, "lora/Llama-output")         # Checkpoint output directory
FINAL_ADAPTER_PATH = os.path.join(OUTPUT_DIR, "Llama-8b-lora-ultra")# Exported LoRA adapter
MERGED_MODEL_PATH = os.path.join(GDRIVE_BASE, "Llama-8b-merged-full")

# Ensure required directories exist
for path in [GDRIVE_BASE, RAG_DATA_DIR, OUTPUT_DIR, FINAL_ADAPTER_PATH, MERGED_MODEL_PATH]:
    os.makedirs(path, exist_ok=True)

# Training configurations for 16GB VRAM constraint
MAX_SEQ_LENGTH = 2048
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8      # Effective batch size = 16
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

print(f"Dataset path: {DATASET_PATH}")
print(f"RAG knowledge base directory: {RAG_DATA_DIR}")
print(f"LoRA adapter destination: {FINAL_ADAPTER_PATH}")




Loads the question-answer dataset directly from the JSONL file on disk and validates the schema.


In [ ]:
from datasets import load_dataset
import os

# Check if dataset exists on disk
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"Training dataset not found at: {DATASET_PATH}\n"
        f"Please place your 'master.jsonl' or 'finetune.jsonl' in {GDRIVE_BASE}/"
    )

print(f"Loading dataset directly from file: {DATASET_PATH}")
raw_dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

# Split into train and validation sets (85% train, 15% validation)
split_dataset = raw_dataset.train_test_split(test_size=0.15, seed=42)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

print(f"Dataset split complete: {len(train_dataset)} training samples, {len(val_dataset)} validation samples.")

# Format examples into standard instruction prompts with EOS tokens
def formatting_func(example):
    instruction = example.get("instruction", "").strip()
    input_text = example.get("input", "").strip()
    output = example.get("output", "").strip()

    if input_text:
        return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Output:\n{output}"
    return f"### Instruction:\n{instruction}\n\n### Output:\n{output}"

print("\nFirst training record formatted preview:")
print("-" * 60)
print(formatting_func(train_dataset[0]))
print("-" * 60)



Loads base weights in NF4 4-bit precision, applies prepare_model_for_kbit_training, and configures LoRA target projections.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# 4-bit quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True
)

# Load tokenizer and set padding token
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load base model in 4-bit
print("Loading 4-bit base model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=compute_dtype
)

# Essential for QLoRA stability and gradient checkpointing
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# LoRA adapter configuration targeting all linear attention and MLP layers
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    bias="none"
)

model = get_peft_model(model, peft_config)
print("\nTrainable parameters summary:")
model.print_trainable_parameters()



Runs training with SFTTrainer, utilizing paged_adamw_8bit optimizer to prevent VRAM spikes during gradient accumulation.


In [ ]:
from transformers import TrainingArguments, DataCollatorForLanguageModeling
from trl import SFTTrainer

# Training configuration
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    warmup_ratio=0.05,
    learning_rate=LEARNING_RATE,
    bf16=(compute_dtype == torch.bfloat16),
    fp16=(compute_dtype == torch.float16),
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=3407,
    dataloader_pin_memory=False,
    report_to="none",
    gradient_checkpointing=True
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    formatting_func=formatting_func,
    max_seq_length=MAX_SEQ_LENGTH,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
        pad_to_multiple_of=8
    ),
    args=training_args
)

# Execute training
print("Starting fine-tuning...")
trainer.train()

# Save LoRA adapter weights and tokenizer
print(f"Saving LoRA adapter to {FINAL_ADAPTER_PATH}")
model.save_pretrained(FINAL_ADAPTER_PATH)
tokenizer.save_pretrained(FINAL_ADAPTER_PATH)
print("LoRA adapter saved successfully.")



Clears trainer state from GPU memory to free VRAM for inference. Also provides an option to merge LoRA into full 16-bit weights.


In [ ]:
import gc
from peft import PeftModel

# Free training memory
print("Clearing training objects from memory...")
try:
    del model
    del trainer
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

# Optional function to merge weights into a standalone 16-bit model
def merge_and_export_model(base_model_name, adapter_path, export_path):
    print(f"Loading base model ({base_model_name}) in 16-bit for clean weight merging...")
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=compute_dtype,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True
    )

    print(f"Loading LoRA adapter from {adapter_path}...")
    peft_model = PeftModel.from_pretrained(base_model, adapter_path)

    print("Merging weights...")
    merged_model = peft_model.merge_and_unload()

    print(f"Saving merged model to {export_path}...")
    merged_model.save_pretrained(export_path, safe_serialization=True, max_shard_size="4GB")
    
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    tokenizer.save_pretrained(export_path)
    print("Merged model export complete.")

    del base_model
    del peft_model
    del merged_model
    gc.collect()
    torch.cuda.empty_cache()

# Flag to control merging: False uses fast 4-bit adapter inference (~6GB VRAM)
ENABLE_MERGE = False
if ENABLE_MERGE:
    merge_and_export_model(MODEL_NAME, FINAL_ADAPTER_PATH, MERGED_MODEL_PATH)
else:
    print("Using direct 4-bit LoRA adapter inference.")



Loads the fine-tuned adapter on top of the 4-bit base model for fast generation.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch
import os

# Inference quantization configuration
inference_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Check whether to load merged model or base model + adapter
if os.path.exists(os.path.join(MERGED_MODEL_PATH, "config.json")):
    load_target = MERGED_MODEL_PATH
    is_merged = True
    print(f"Loading merged model from: {load_target}")
else:
    load_target = MODEL_NAME
    is_merged = False
    print(f"Loading base model: {load_target} with adapter: {FINAL_ADAPTER_PATH}")

eval_model = AutoModelForCausalLM.from_pretrained(
    load_target,
    quantization_config=inference_bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=compute_dtype
)
eval_tokenizer = AutoTokenizer.from_pretrained(load_target)
eval_tokenizer.pad_token = eval_tokenizer.eos_token

# Attach adapter if loading from base model
if not is_merged and os.path.exists(os.path.join(FINAL_ADAPTER_PATH, "adapter_config.json")):
    print("Attaching LoRA adapter weights...")
    eval_model = PeftModel.from_pretrained(eval_model, FINAL_ADAPTER_PATH)

print("Inference model ready.")

# Shared helper function to invoke the model
def run_academic_llm(instruction: str, context: str = "", max_tokens: int = 700, temperature: float = 0.5):
    if context:
        prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{context}\n\n### Output:\n"
    else:
        prompt = f"### Instruction:\n{instruction}\n\n### Output:\n"

    inputs = eval_tokenizer(prompt, return_tensors="pt").to(eval_model.device)

    with torch.no_grad():
        outputs = eval_model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            top_p=0.9,
            do_sample=(temperature > 0.1),
            pad_token_id=eval_tokenizer.eos_token_id,
            eos_token_id=eval_tokenizer.eos_token_id
        )

    full_output = eval_tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "### Output:\n" in full_output:
        return full_output.split("### Output:\n")[-1].strip()
    return full_output.strip()




Semantic indexing and retrieval engine using sentence-transformers/all-MiniLM-L6-v2.
Supports document chunking, cosine similarity ranking, and multi-tenant metadata filtering (subject, institution, level, unit).


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import os
import pypdf

class AcademicRAGIndex:
    """Vector store and semantic search engine with metadata filtering."""
    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        print(f"Initializing embedding model: {model_name}")
        self.encoder = SentenceTransformer(model_name)
        self.documents = []    # Text chunks
        self.metadata = []     # Associated metadata dictionaries
        self.embeddings = None # Vector array

    def add_document(self, text: str, meta: dict, chunk_size: int = 400, overlap: int = 50):
        """Splits text into overlapping word chunks and updates the vector index."""
        words = text.split()
        if not words:
            return

        chunks = []
        i = 0
        while i < len(words):
            chunk = " ".join(words[i:i + chunk_size])
            chunks.append(chunk)
            i += (chunk_size - overlap)

        vectors = self.encoder.encode(chunks, convert_to_numpy=True, normalize_embeddings=True)

        for c, m in zip(chunks, [meta] * len(chunks)):
            self.documents.append(c)
            self.metadata.append(m)

        if self.embeddings is None:
            self.embeddings = vectors
        else:
            self.embeddings = np.vstack([self.embeddings, vectors])

    def search(self, query: str, top_k: int = 3, subject: str = None, institution: str = None, education_level: str = None, unit: str = None):
        """Retrieves top-k chunks matching the query, filtered by metadata."""
        if self.embeddings is None or len(self.documents) == 0:
            return []

        # Filter candidate indices based on metadata
        candidates = []
        for idx, meta in enumerate(self.metadata):
            match = True
            if subject and meta.get("subject", "").lower() != subject.lower():
                match = False
            if institution and institution.lower() != "universal" and meta.get("institution", "").lower() not in [institution.lower(), "universal"]:
                match = False
            if education_level and meta.get("education_level", "").lower() != education_level.lower():
                match = False
            if unit and meta.get("unit", "").lower() != unit.lower():
                match = False
            if match:
                candidates.append(idx)

        # Fallback to all documents if filter yields zero matches
        if not candidates:
            candidates = list(range(len(self.documents)))

        # Compute cosine similarity
        query_vec = self.encoder.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
        candidate_vecs = self.embeddings[candidates]
        scores = np.dot(candidate_vecs, query_vec)

        # Rank results
        ranked_indices = np.argsort(scores)[::-1][:top_k]
        results = []
        for rank in ranked_indices:
            orig_idx = candidates[rank]
            results.append({
                "content": self.documents[orig_idx],
                "metadata": self.metadata[orig_idx],
                "score": float(scores[rank])
            })
        return results

# Initialize vector index
rag_index = AcademicRAGIndex(EMBEDDING_MODEL_NAME)
print("Vector index initialized.")



Scans RAG_DATA_DIR and ingests all curriculum files (.txt, .md, .pdf, .json) directly from disk.
No synthetic in-code data is generated; the index is built strictly from your uploaded files.


In [ ]:
import os
import json
import pypdf

def ingest_directory_to_rag(rag_index, directory_path):
    """Reads files directly from directory and adds them to RAG vector store."""
    if not os.path.exists(directory_path):
        print(f"Directory not found: {directory_path}")
        return 0

    files_indexed = 0
    supported_extensions = ('.txt', '.md', '.pdf', '.json')
    file_list = [f for f in os.listdir(directory_path) if f.lower().endswith(supported_extensions)]

    if not file_list:
        print(f"No documents found in {directory_path}. Upload your syllabus or notes files here.")
        return 0

    print(f"Found {len(file_list)} files in {directory_path} to index.")

    for filename in file_list:
        file_path = os.path.join(directory_path, filename)
        file_ext = os.path.splitext(filename)[1].lower()
        
        # Derive subject/title from filename conventions (e.g. 'DBMS_Unit3_Transactions.txt')
        base_title = os.path.splitext(filename)[0]
        meta = {
            "title": base_title,
            "filename": filename,
            "subject": "General",
            "institution": "Universal",
            "education_level": "Undergraduate",
            "unit": "General",
            "doc_type": file_ext.replace('.', '').upper()
        }

        # Try to parse subject from filename prefix if separated by underscore or dash
        if "_" in base_title:
            parts = base_title.split("_")
            meta["subject"] = parts[0]
            if len(parts) > 1:
                meta["unit"] = parts[1]

        try:
            if file_ext in ('.txt', '.md'):
                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    content = f.read()
                rag_index.add_document(content, meta)
                files_indexed += 1
                print(f"Indexed text file: {filename}")

            elif file_ext == '.pdf':
                reader = pypdf.PdfReader(file_path)
                pdf_text = ""
                for page_idx, page in enumerate(reader.pages):
                    page_text = page.extract_text()
                    if page_text:
                        pdf_text += f"\n--- Page {page_idx + 1} ---\n" + page_text
                if pdf_text.strip():
                    rag_index.add_document(pdf_text, meta)
                    files_indexed += 1
                    print(f"Indexed PDF file: {filename} ({len(reader.pages)} pages)")

            elif file_ext == '.json':
                with open(file_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                # Handle either a list of sections or a structured syllabus dictionary
                if isinstance(data, list):
                    for item in data:
                        text_val = item.get("content") or item.get("text") or str(item)
                        item_meta = {**meta, **{k: v for k, v in item.items() if k not in ("content", "text")}}
                        rag_index.add_document(text_val, item_meta)
                elif isinstance(data, dict):
                    text_val = data.get("content") or json.dumps(data, indent=2)
                    rag_index.add_document(text_val, {**meta, **data.get("metadata", {})})
                files_indexed += 1
                print(f"Indexed JSON curriculum file: {filename}")

        except Exception as e:
            print(f"Error processing {filename}: {e}")

    print(f"\nIngestion complete: {files_indexed} files loaded. Total chunks in index: {len(rag_index.documents)}")
    return files_indexed

# Run file-based ingestion
ingest_directory_to_rag(rag_index, RAG_DATA_DIR)




Retrieves course documents from the knowledge base and formats answers to strictly adhere to the requested mark weightage (2, 5, 10, or 15 marks).


In [ ]:
def generate_syllabus_notes(question: str, marks: int = 5, subject: str = "General", institution: str = "Universal", education_level: str = "Undergraduate"):
    """Generates exam notes matching syllabus scope retrieved from files."""
    # Retrieve relevant references from file-based vector index
    retrieved_chunks = rag_index.search(
        query=question,
        top_k=2,
        subject=subject if subject != "General" else None,
        institution=institution,
        education_level=education_level
    )
    context_str = "\n\n".join([f"[Source: {c['metadata']['title']}]\n{c['content']}" for c in retrieved_chunks])

    # Mark guidelines
    guidelines = {
        2: "Provide a 1-sentence definition and 2 clear bullet points. Maximum 120 words.",
        5: "Provide a concise definition, 4-5 structured bullet points with sub-headings, and a concrete example. Maximum 300 words.",
        10: "Provide an academic 10-mark structure: (1) Formal Definition, (2) Core Architecture/Properties breakdown, (3) Detailed Mechanism, (4) Concrete Real-World Example. Maximum 600 words.",
        15: "Provide an in-depth essay answer: (1) Introduction, (2) Theoretical Mechanism & Architecture, (3) Comparison Table/Trade-offs, (4) Practical Case Study, (5) Conclusion. Maximum 900 words."
    }
    rubric_hint = guidelines.get(marks, f"Format response cleanly for {marks} marks.")
    token_budget = {2: 180, 5: 380, 10: 750, 15: 1100}.get(marks, 600)

    instruction = (
        f"You are a university instructor in {subject} at {institution} ({education_level} level).\n"
        f"Answer the exam question: '{question}'.\n"
        f"Mark Allocated: {marks} Marks.\n"
        f"Grading Guideline: {rubric_hint}\n"
        f"Constraint: Base the answer strictly on the provided syllabus and reference text."
    )

    print(f"Retrieved {len(retrieved_chunks)} references for query: '{question}'")
    return run_academic_llm(instruction=instruction, context=context_str, max_tokens=token_budget, temperature=0.4)

# Example usage
print("-" * 60)
print("Example: 10-Mark Answer Generation")
print("-" * 60)
print(generate_syllabus_notes(
    question="Explain the ACID properties in DBMS with real-world banking transactions.",
    marks=10,
    subject="DBMS"
))




Extracts syllabus topics from the indexed files to construct a balanced university question set with model answer keys.


In [ ]:
def generate_exam_paper(topic: str, subject: str = "General", institution: str = "Universal", education_level: str = "Undergraduate", include_solutions: bool = True):
    """Generates examination questions with marking schemes from indexed course documents."""
    retrieved_chunks = rag_index.search(
        query=topic,
        top_k=2,
        subject=subject if subject != "General" else None,
        institution=institution,
        education_level=education_level
    )
    context_str = "\n\n".join([f"[{c['metadata']['title']}]\n{c['content']}" for c in retrieved_chunks])

    instruction = (
        f"Create a balanced university examination question set for {subject} on '{topic}'.\n"
        f"Format:\n"
        f"1. Part A (2 Marks each): 2 Short definition questions.\n"
        f"2. Part B (5 or 10 Marks): 1 Analytical or mechanism question.\n"
        f"3. Part C (15 Marks): 1 Comprehensive case-study or system design question.\n"
        f"Target Level: {education_level} ({institution}).\n"
    )
    if include_solutions:
        instruction += "Provide a model answer key and key required points for full marks under each question."

    return run_academic_llm(instruction=instruction, context=context_str, max_tokens=900, temperature=0.6)

# Example usage
print("-" * 60)
print("Example: Question Paper Generation")
print("-" * 60)
print(generate_exam_paper(topic="Deadlocks and Banker's Algorithm", subject="Operating Systems"))




Evaluates student answers by comparing them directly against the ground truth indexed in the RAG store.


In [ ]:
def evaluate_student_answer(question: str, student_answer: str, max_marks: int = 10, subject: str = "General"):
    """Grades student submissions against source material from the RAG store."""
    retrieved = rag_index.search(query=question, top_k=2, subject=subject if subject != "General" else None)
    ground_truth = "\n\n".join([c["content"] for c in retrieved])

    instruction = (
        f"You are grading an academic exam answer in {subject}.\n"
        f"Question: '{question}' (Total Marks: {max_marks})\n\n"
        f"Student Answer:\n\"\"{student_answer}\"\"\n\n"
        f"Evaluate the submission against the reference material.\n"
        f"Format:\n"
        f"1. Score Awarded: [X / {max_marks} Marks]\n"
        f"2. Concepts Correctly Covered: [What the student understood]\n"
        f"3. Missing Points: [Key definitions, formulas, or steps omitted]\n"
        f"4. Improvement Feedback: [Specific steps to gain full marks]"
    )

    context = f"Authoritative Reference:\n{ground_truth}"
    return run_academic_llm(instruction=instruction, context=context, max_tokens=650, temperature=0.3)

# Example evaluation
sample_answer = """
ACID stands for Atomicity, Consistency, Isolation, and Durability.
Atomicity means transactions cannot be divided, so either it completes or fails.
Consistency means the data is always correct.
Isolation means transactions don't bother each other.
Durability means when saved, data is permanently in memory.
For example, if you send money from Bank A to B, money shouldn't disappear.
"""

print("-" * 60)
print("Example: Automated Answer Evaluation")
print("-" * 60)
print(evaluate_student_answer(
    question="Explain the ACID properties in DBMS with real-world banking examples.",
    student_answer=sample_answer,
    max_marks=10,
    subject="DBMS"
))




Synthesizes high-yield revision flashcards, quick cheat-sheets, and diagnostic recovery schedules based on weak topics.


In [ ]:
def generate_study_materials(topic: str, material_type: str = "cheat_sheet", subject: str = "General"):
    """Generates study materials (cheat_sheet or flashcards) from indexed notes."""
    retrieved = rag_index.search(query=topic, top_k=2, subject=subject if subject != "General" else None)
    context = "\n\n".join([c["content"] for c in retrieved])

    if material_type == "flashcards":
        instruction = (
            f"Generate 5 active recall flashcards for {subject} on '{topic}'.\n"
            f"Format:\n"
            f"Card X:\n"
            f"Question: [Core exam question]\n"
            f"Answer: [Concise 1-2 sentence answer with key terms]"
        )
    else:
        instruction = (
            f"Create a 5-minute pre-exam cheat-sheet for {subject} on '{topic}'.\n"
            f"Include: (1) Core Definitions, (2) Comparison Table, (3) Common Pitfalls to Avoid."
        )

    return run_academic_llm(instruction=instruction, context=context, max_tokens=700, temperature=0.4)

def recommend_adaptive_learning_path(weak_topics: list, subject: str = "General", current_level: str = "Undergraduate"):
    """Generates a diagnostic remedial study plan based on student knowledge gaps."""
    weak_str = ", ".join(weak_topics)
    instruction = (
        f"A student in {current_level} {subject} needs remediation in: {weak_str}.\n"
        f"Create an adaptive 3-day recovery study plan:\n"
        f"- Day 1: Foundational concepts and definitions\n"
        f"- Day 2: Detailed mechanisms and solved examples\n"
        f"- Day 3: Practice exam questions with self-check tests"
    )
    return run_academic_llm(instruction=instruction, context="", max_tokens=600, temperature=0.5)

# Example usage
print("-" * 60)
print("Example: Revision Flashcards")
print("-" * 60)
print(generate_study_materials(topic="Process Scheduling and Deadlocks", material_type="flashcards", subject="Operating Systems"))



A clean multi-turn Telegram bot interface for students to access syllabus explanations, question papers, answer grading, and revision sheets.


In [ ]:
import os
import telebot
from threading import Thread
from transformers import TextIteratorStreamer

# Telegram Bot Token from environment variable or direct input
TELEGRAM_BOT_TOKEN = os.getenv("TELEGRAM_API_KEY", "8357868813:AAFeDCX3E1sQqFAbOoAd-LPBVVCe8YZGAHI")
bot = telebot.TeleBot(TELEGRAM_BOT_TOKEN)

# In-memory session tracking for active user dialogs
user_profiles = {}  # chat_id -> {'subject': ..., 'institution': ..., 'level': ...}
user_flows = {}     # chat_id -> flow state data

def get_profile(chat_id):
    if chat_id not in user_profiles:
        user_profiles[chat_id] = {'subject': 'General', 'institution': 'Universal', 'level': 'Undergraduate'}
    return user_profiles[chat_id]

# Command: /start, /help
@bot.message_handler(commands=['start', 'help'])
def send_welcome(message):
    chat_id = message.chat.id
    prof = get_profile(chat_id)
    text = (
        "Welcome to the Academic Assistant.\n\n"
        f"Active Profile: {prof['subject']} | {prof['institution']} | {prof['level']}\n\n"
        "Commands:\n"
        "/ask - Ask an exam question and choose marks (2, 5, 10, 15)\n"
        "/eval - Submit an answer for grading and gap analysis\n"
        "/quiz - Generate exam questions and answer keys\n"
        "/notes - Generate quick revision cheat-sheets\n"
        "/profile - Update your subject, institution, or level\n"
        "/cancel - Abort current command"
    )
    bot.reply_to(message, text)

# Command: /cancel
@bot.message_handler(commands=['cancel'])
def cancel_action(message):
    chat_id = message.chat.id
    if chat_id in user_flows:
        del user_flows[chat_id]
        bot.reply_to(message, "Active session cancelled. Use /help to see commands.")
    else:
        bot.reply_to(message, "No active session to cancel.")

# Command: /profile
@bot.message_handler(commands=['profile'])
def configure_profile(message):
    chat_id = message.chat.id
    user_flows[chat_id] = {'action': 'set_profile'}
    text = (
        "Configure your academic profile.\n"
        "Reply with: Subject, Institution, Education Level\n\n"
        "Example: DBMS, Anna University, Undergraduate\n"
        "Example: Physics, CBSE, High School"
    )
    bot.reply_to(message, text)
    bot.register_next_step_handler(message, save_profile_step)

def save_profile_step(message):
    chat_id = message.chat.id
    parts = [p.strip() for p in message.text.split(",")]
    if len(parts) >= 3:
        user_profiles[chat_id] = {
            'subject': parts[0],
            'institution': parts[1],
            'level': parts[2]
        }
        bot.send_message(chat_id, f"Profile updated: {parts[0]} ({parts[1]} - {parts[2]})")
    else:
        bot.send_message(chat_id, "Invalid format. Expected: Subject, Institution, Education Level")
    if chat_id in user_flows:
        del user_flows[chat_id]

# Command: /ask
@bot.message_handler(commands=['ask'])
def handle_ask(message):
    chat_id = message.chat.id
    user_flows[chat_id] = {'action': 'ask'}
    bot.reply_to(message, "Enter your question or topic:")
    bot.register_next_step_handler(message, ask_step_question)

def ask_step_question(message):
    chat_id = message.chat.id
    if message.text.startswith('/'):
        return
    user_flows[chat_id]['question'] = message.text
    bot.send_message(chat_id, "Specify marks (2, 5, 10, or 15):")
    bot.register_next_step_handler(message, ask_step_marks)

def ask_step_marks(message):
    chat_id = message.chat.id
    if chat_id not in user_flows or 'question' not in user_flows[chat_id]:
        bot.send_message(chat_id, "Session expired. Use /ask to start again.")
        return

    try:
        marks = int(message.text.strip())
    except ValueError:
        marks = 5

    question = user_flows[chat_id]['question']
    prof = get_profile(chat_id)

    retrieved = rag_index.search(
        query=question,
        top_k=2,
        subject=prof['subject'] if prof['subject'] != 'General' else None,
        institution=prof['institution'],
        education_level=prof['level']
    )
    context_str = "\n\n".join([f"[{c['metadata']['title']}]\n{c['content']}" for c in retrieved])

    token_budget = {2: 180, 5: 380, 10: 750, 15: 1100}.get(marks, 600)
    prompt = (
        f"### Instruction:\n"
        f"You are an instructor in {prof['subject']} ({prof['institution']}).\n"
        f"Answer the question: '{question}' for {marks} marks using academic structure.\n\n"
        f"### Input:\n{context_str}\n\n"
        f"### Output:\n"
    )

    stream_llm_to_telegram(chat_id, prompt, max_tokens=token_budget)
    del user_flows[chat_id]

# Command: /eval
@bot.message_handler(commands=['eval'])
def handle_eval(message):
    chat_id = message.chat.id
    user_flows[chat_id] = {'action': 'eval'}
    bot.reply_to(message, "What was the exam question?")
    bot.register_next_step_handler(message, eval_step_question)

def eval_step_question(message):
    chat_id = message.chat.id
    if message.text.startswith('/'):
        return
    user_flows[chat_id]['question'] = message.text
    bot.send_message(chat_id, "Enter total marks for this question (e.g., 5, 10):")
    bot.register_next_step_handler(message, eval_step_marks)

def eval_step_marks(message):
    chat_id = message.chat.id
    try:
        user_flows[chat_id]['marks'] = int(message.text.strip())
    except ValueError:
        user_flows[chat_id]['marks'] = 10
    bot.send_message(chat_id, "Paste your answer to evaluate:")
    bot.register_next_step_handler(message, eval_step_answer)

def eval_step_answer(message):
    chat_id = message.chat.id
    if chat_id not in user_flows:
        return

    question = user_flows[chat_id]['question']
    marks = user_flows[chat_id]['marks']
    student_ans = message.text
    prof = get_profile(chat_id)

    retrieved = rag_index.search(
        query=question,
        top_k=2,
        subject=prof['subject'] if prof['subject'] != 'General' else None
    )
    ground_truth = "\n\n".join([c["content"] for c in retrieved])

    prompt = (
        f"### Instruction:\n"
        f"Grade this student answer for {marks} marks in {prof['subject']}.\n"
        f"Question: '{question}'\n"
        f"Student Answer: \"{student_ans}\"\n\n"
        f"Provide: Score Awarded [X/{marks}], Strengths, Missing Points, and Improvement Advice.\n\n"
        f"### Input:\n{ground_truth}\n\n"
        f"### Output:\n"
    )

    stream_llm_to_telegram(chat_id, prompt, max_tokens=650)
    del user_flows[chat_id]

# Command: /quiz
@bot.message_handler(commands=['quiz'])
def handle_quiz(message):
    chat_id = message.chat.id
    prof = get_profile(chat_id)
    topic = message.text.replace('/quiz', '').strip()
    if not topic:
        topic = "Core syllabus topics"

    retrieved = rag_index.search(
        query=topic,
        top_k=2,
        subject=prof['subject'] if prof['subject'] != 'General' else None
    )
    context = "\n\n".join([c["content"] for c in retrieved])

    prompt = (
        f"### Instruction:\n"
        f"Generate a balanced exam question set (Part A 2-mark, Part B 5/10-mark, Part C 15-mark) with answer keys for {prof['subject']} on '{topic}'.\n\n"
        f"### Input:\n{context}\n\n"
        f"### Output:\n"
    )
    stream_llm_to_telegram(chat_id, prompt, max_tokens=850)

# Command: /notes
@bot.message_handler(commands=['notes'])
def handle_notes(message):
    chat_id = message.chat.id
    prof = get_profile(chat_id)
    topic = message.text.replace('/notes', '').strip()
    if not topic:
        topic = "Summary review"

    retrieved = rag_index.search(
        query=topic,
        top_k=2,
        subject=prof['subject'] if prof['subject'] != 'General' else None
    )
    context = "\n\n".join([c["content"] for c in retrieved])

    prompt = (
        f"### Instruction:\n"
        f"Create a high-yield 5-minute pre-exam cheat-sheet with definitions, key differences, and exam tips for {prof['subject']} on '{topic}'.\n\n"
        f"### Input:\n{context}\n\n"
        f"### Output:\n"
    )
    stream_llm_to_telegram(chat_id, prompt, max_tokens=700)

# Streaming response utility to Telegram
def stream_llm_to_telegram(chat_id, prompt, max_tokens=600):
    wait_msg = bot.send_message(chat_id, "Searching course references and generating response...")
    streaming_msg = bot.send_message(chat_id, "...")
    bot.delete_message(chat_id, wait_msg.message_id)

    inputs = eval_tokenizer(prompt, return_tensors="pt").to(eval_model.device)
    streamer = TextIteratorStreamer(eval_tokenizer, skip_prompt=True, skip_special_tokens=True)

    generation_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=max_tokens,
        temperature=0.4,
        top_p=0.9,
        do_sample=True,
        pad_token_id=eval_tokenizer.eos_token_id
    )

    thread = Thread(target=eval_model.generate, kwargs=generation_kwargs)
    thread.start()

    full_response = ""
    update_interval = 14
    token_counter = 0

    for new_text in streamer:
        full_response += new_text
        token_counter += 1

        if token_counter % update_interval == 0:
            try:
                bot.edit_message_text(full_response + " |", chat_id, streaming_msg.message_id)
            except Exception:
                pass

    try:
        bot.edit_message_text(full_response, chat_id, streaming_msg.message_id)
    except Exception:
        pass

print("Telegram bot polling started. Message your bot on Telegram to interact.")
bot.infinity_polling()
